In [1]:
import tifffile as tiff
from PIL import Image
import os
from glob import glob
import cv2
import json
from math import ceil
import tqdm
import numpy as np
from matplotlib import pyplot as plt
################### CFGS ###############
SCALE = 2
root = '/mnt/d/AGGC2022/AGGC2022/AGGC2022_train/Subset1_Train_image'
anno = '/mnt/d/AGGC2022/AGGC2022/AGGC2022_train/Subset1_Train_annotation'

train_data_manifest = [os.path.basename(item).replace('.png', '') for item in json.load(open('/mnt/d/Cpathdata/data/aggc_splits_json/train_data.json', 'r'))]
val_data_manifest = [os.path.basename(item).replace('.png', '') for item in json.load(open('/mnt/d/Cpathdata/data/aggc_splits_json/validation_data.json', 'r'))]


save_dir = '/mnt/f/aggc_10x_1024_reproduce'
os.makedirs(os.path.join(save_dir, 'images', 'train'), exist_ok=True)
os.makedirs(os.path.join(save_dir, 'annotations', 'train'), exist_ok=True)
os.makedirs(os.path.join(save_dir, 'images', 'validation'), exist_ok=True)
os.makedirs(os.path.join(save_dir, 'annotations', 'validation'), exist_ok=True)
# os.makedirs(os.path.join(save_dir, 'tissue_masks', 'train'), exist_ok=True)
# os.makedirs(os.path.join(save_dir, 'tissue_masks', 'validation'), exist_ok=True)
tile_size = 1024
stride = 512
rgb_thresh = 215
ratio = 0.5
########################################



In [2]:
################ train data ###############
Image.MAX_IMAGE_PIXELS = None
for file in tqdm.tqdm(train_data_manifest):
    base_filename = file
    image = tiff.imread(os.path.join(root, file+'.tiff'))
    H, W = image.shape[:-1]
    h, w = ceil(H/SCALE), ceil(W/SCALE)
    image = cv2.resize(image, dsize=(w, h), interpolation=cv2.INTER_AREA)
   

    tumor_mask = np.zeros([H, W], dtype=np.uint8)
  
    annopath = glob(os.path.join(anno, file, 'G[3-5]_Mask.tif'))
    for fp in annopath:
        mask = Image.open(fp)
        mask = np.array(mask).astype(np.uint8)
        tumor_mask[mask>0] = 1
    tumor_mask = cv2.resize(tumor_mask, dsize=(w, h), interpolation=cv2.INTER_NEAREST)


    if h < tile_size:
        y_starts = [0]
    else:
        y_starts = list(range(0, h - tile_size + 1, stride))
        if (h - tile_size) % stride != 0:
            y_starts.append(h - tile_size)

    if w < tile_size:
        x_starts = [0]
    else:
        x_starts = list(range(0, w - tile_size + 1, stride))
        if (w - tile_size) % stride != 0:
            x_starts.append(w - tile_size)

    for y0 in y_starts:
        for x0 in x_starts:
            region_h, region_w = slice(y0, y0 + tile_size), slice(x0, x0 + tile_size)                           
            img  = image[region_h, region_w, ...]
            mask  = tumor_mask[region_h, region_w, ...]

            bg = (img[...,0] >= rgb_thresh) & (img[...,1] >= rgb_thresh) & (img[...,2] >= rgb_thresh)
            fg = ~bg 
            if np.mean(fg) < ratio: 
                continue
            if np.sum(mask) == 0:
                continue
            filename = f"{base_filename},x={x0*SCALE},y={y0*SCALE}.png"

            Image.fromarray(img).save(os.path.join(save_dir, 'images', 'train', filename))
            mask = Image.fromarray(mask.astype(np.uint8), mode='L')
            mask.putpalette([0, 0, 0, 255, 255, 255] + [0, 0, 0] * (256-2))
            mask.save(os.path.join(save_dir, 'annotations', 'train', filename))

  0%|          | 0/93 [00:00<?, ?it/s]/tmp/ipykernel_213403/2425222650.py:50: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  mask = Image.fromarray(mask.astype(np.uint8), mode='L')
100%|██████████| 93/93 [3:44:40<00:00, 144.95s/it]  


In [3]:
################ val data ###############
Image.MAX_IMAGE_PIXELS = None
for file in tqdm.tqdm(val_data_manifest):
    base_filename = file
    image = tiff.imread(os.path.join(root, file+'.tiff'))
    H, W = image.shape[:-1]
    h, w = ceil(H/SCALE), ceil(W/SCALE)
    image = cv2.resize(image, dsize=(w, h), interpolation=cv2.INTER_AREA)
   

    tumor_mask = np.zeros([H, W], dtype=np.uint8)
  
    annopath = glob(os.path.join(anno, file, 'G[3-5]_Mask.tif'))
    for fp in annopath:
        mask = Image.open(fp)
        mask = np.array(mask).astype(np.uint8)
        tumor_mask[mask>0] = 1
    tumor_mask = cv2.resize(tumor_mask, dsize=(w, h), interpolation=cv2.INTER_NEAREST)


    if h < tile_size:
        y_starts = [0] 
    else:
        y_starts = list(range(0, h - tile_size + 1, tile_size))
        if (h - tile_size) % tile_size != 0:                                                                                                              
            y_starts.append(h - tile_size)

    if w < tile_size:
        x_starts = [0]
    else:
        x_starts = list(range(0, w - tile_size + 1, tile_size))
        if (w - tile_size) % tile_size != 0:
            x_starts.append(w - tile_size)

    for y0 in y_starts:
        for x0 in x_starts:
            region_h, region_w = slice(y0, y0 + tile_size), slice(x0, x0 + tile_size)                           
            img  = image[region_h, region_w, ...]
            mask  = tumor_mask[region_h, region_w, ...]

            bg = (img[...,0] >= rgb_thresh) & (img[...,1] >= rgb_thresh) & (img[...,2] >= rgb_thresh)
            fg = ~bg 
            if np.mean(fg) < ratio: 
                continue
            if np.sum(mask) == 0:
                continue
            filename = f"{base_filename},x={x0*SCALE},y={y0*SCALE}.png"
            mask = Image.fromarray(mask)
            mask.putpalette([0, 0, 0, 255, 255, 255] + (256-2) * [0, 0, 0])
            Image.fromarray(img).save(os.path.join(save_dir, 'images', 'validation', filename))
            mask.save(os.path.join(save_dir, 'annotations', 'validation', filename))


100%|██████████| 12/12 [12:16<00:00, 61.38s/it]


In [4]:
import PIL
PIL.Image.MAX_IMAGE_PIXELS = None
os.makedirs(os.path.join(save_dir, 'images', 'test'), exist_ok=True)
os.makedirs(os.path.join(save_dir, 'annotations', 'test'), exist_ok=True)

root = '/mnt/d/AGGC2022/AGGC2022/AGGC2022_test/Subset1_Test_image'
anno = '/mnt/d/AGGC2022/AGGC2022/AGGC2022_test/Subset1_Test_annotation'

files = glob(os.path.join(root, '*.tiff'))

for file in tqdm.tqdm(files):
    base_filename = os.path.basename(file).replace('.tiff', '')
    img = tiff.imread(file)
    H, W = img.shape[:-1]
    tumor_mask = np.zeros([H, W], dtype=np.uint8)

    h, w = ceil(H/SCALE), ceil(W/SCALE)
    img = cv2.resize(img, dsize=(w, h), interpolation=cv2.INTER_AREA)

    annopath = glob(os.path.join(anno, os.path.basename(file).replace('.tiff', ''), 'G[3-5]_Mask.tif'))
    for fp in annopath:
        mask = Image.open(fp)
        mask = np.array(mask).astype(np.uint8)
        tumor_mask[mask>0] = 1
    tumor_mask = cv2.resize(tumor_mask, dsize=(w, h) , interpolation=cv2.INTER_NEAREST)
    # tumor_mask = Image.fromarray(tumor_mask)

    if h < tile_size:
        y_starts = [0]
    else:
        y_starts = list(range(0, h - tile_size + 1, tile_size))
        if (h - tile_size) % tile_size != 0:
            y_starts.append(h - tile_size)

    if w < tile_size:
        x_starts = [0]
    else:
        x_starts = list(range(0, w - tile_size + 1, tile_size))
        if (w - tile_size) % tile_size != 0:
            x_starts.append(w - tile_size)

    for y0 in y_starts:
        for x0 in x_starts:
            region_h, region_w = slice(y0, y0 + tile_size), slice(x0, x0 + tile_size)                           
            image  = img[region_h, region_w, ...]
            mask  = tumor_mask[region_h, region_w, ...]

            bg = (image[...,0] >= rgb_thresh) & (image[...,1] >= rgb_thresh) & (image[...,2] >= rgb_thresh)
            fg = ~bg 
            if np.mean(fg) < ratio: 
                continue
            filename = f"{base_filename},x={x0*SCALE},y={y0*SCALE}.png"
            mask = Image.fromarray(mask)
            mask.putpalette([0, 0, 0, 255, 255, 255] + (256-2) * [0, 0, 0])
            Image.fromarray(image).save(os.path.join(save_dir, 'images', 'test', filename))
            mask.save(os.path.join(save_dir, 'annotations', 'test', filename))

100%|██████████| 45/45 [1:57:23<00:00, 156.53s/it]


In [ ]:
################ train data ###############
stride = tile_size
save_dir = '/mnt/f/aggc_10x_1024_reproduce/'
Image.MAX_IMAGE_PIXELS = None
for file in tqdm.tqdm(train_data_manifest):
    base_filename = file
    image = tiff.imread(os.path.join(root, file+'.tiff'))
    H, W = image.shape[:-1]
    h, w = ceil(H/SCALE), ceil(W/SCALE)
    image = cv2.resize(image, dsize=(w, h), interpolation=cv2.INTER_AREA)
   

    tumor_mask = np.zeros([H, W], dtype=np.uint8)
  
    annopath = glob(os.path.join(anno, file, 'G[3-5]_Mask.tif'))
    for fp in annopath:
        mask = Image.open(fp)
        mask = np.array(mask).astype(np.uint8)
        tumor_mask[mask>0] = 1
    tumor_mask = cv2.resize(tumor_mask, dsize=(w, h), interpolation=cv2.INTER_NEAREST)


    if h < tile_size:
        y_starts = [0]
    else:
        y_starts = list(range(0, h - tile_size + 1, stride))
        if (h - tile_size) % stride != 0:
            y_starts.append(h - tile_size)

    if w < tile_size:
        x_starts = [0]
    else:
        x_starts = list(range(0, w - tile_size + 1, stride))
        if (w - tile_size) % stride != 0:
            x_starts.append(w - tile_size)

    for y0 in y_starts:
        for x0 in x_starts:
            region_h, region_w = slice(y0, y0 + tile_size), slice(x0, x0 + tile_size)                           
            img  = image[region_h, region_w, ...]
            mask  = tumor_mask[region_h, region_w, ...]

            bg = (img[...,0] >= rgb_thresh) & (img[...,1] >= rgb_thresh) & (img[...,2] >= rgb_thresh)
            fg = ~bg 
            if np.mean(fg) < ratio: 
                continue
            # if np.sum(mask) == 0:
            #     continue
            filename = f"{base_filename},x={x0*SCALE},y={y0*SCALE}.png"

            Image.fromarray(img).save(os.path.join(save_dir, 'images', 'train', filename))
            mask = Image.fromarray(mask.astype(np.uint8), mode='L')
            mask.putpalette([0, 0, 0, 255, 255, 255] + [0, 0, 0] * (256-2))
            mask.save(os.path.join(save_dir, 'annotations', 'train', filename))

  0%|          | 0/93 [00:00<?, ?it/s]/tmp/ipykernel_61873/2932623747.py:52: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  mask = Image.fromarray(mask.astype(np.uint8), mode='L')
100%|██████████| 93/93 [1:48:23<00:00, 69.93s/it]


In [ ]:
################## TRAIN DATA ########################

# masks = glob(os.path.join('./data/aggc_10x_tiles/annotations/train', '*.png'))
# count = len(masks)
# pos = 0
# neg = 0

# for m in masks:
#     img = Image.open(m)
#     img = np.array(img)
    
#     if np.sum(img)>0:
#         pos+=1
#     else:
#         neg+=1

# print(f'number of tiles:{count}, positive tiles:{pos}, negative tiles:{neg}')

number of tiles:114649, positive tiles:30017, negative tiles:84632
